In [1]:
# Install pyBKT
%pip install pyBKT


[notice] A new release of pip available: 22.3.1 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyBKT
from pyBKT.models import Model, Roster
import pickle
import random

num_students = 10000
num_attempts = 10000000

data = {
    'student_id': [random.randint(1, num_students) for _ in range(num_attempts)],
    'topicSlug': [random.choice([
        'data-structures', 'algorithms', 'operating-systems', 'networking',
        'database-systems', 'software-engineering', 'web-development',
        'object-oriented-programming', 'machine-learning', 'cloud-computing',
        'cybersecurity', 'mobile-development', 'data-analytics', 'software-testing',
        'devops', 'design-patterns', 'algorithms-optimization',
        'advanced-database-systems', 'distributed-systems', 'artificial-intelligence',
        'network-security', 'blockchain', 'user-experience-design', 'ethical-hacking',
        'software-architecture', 'quantum-computing', 'big-data'
    ]) for _ in range(num_attempts)],
    'isCorrect': [random.randint(0, 1) for _ in range(num_attempts)]
}


df = pd.DataFrame(data)

In [ ]:
# Initialize the model
model = Model(parallel=True, num_fits=1, seed=42, defaults={'skill_name': 'topicSlug', 'correct': 'isCorrect', 'user_id': 'student_id'})

# Initialize the roster model
skills = [
    'data-structures', 'algorithms', 'operating-systems', 'networking',
    'database-systems', 'software-engineering', 'web-development',
    'object-oriented-programming', 'machine-learning', 'cloud-computing',
    'cybersecurity', 'mobile-development', 'data-analytics', 'software-testing',
    'devops', 'design-patterns', 'algorithms-optimization',
    'advanced-database-systems', 'distributed-systems', 'artificial-intelligence',
    'network-security', 'blockchain', 'user-experience-design', 'ethical-hacking',
    'software-architecture', 'quantum-computing', 'big-data'
]

roster = Roster(0, skills, 0.95, False, model)

model.fit(data=df)

with open('computer_science_bktmodel.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('computer_science_roster_model.pkl', 'wb') as f:
    pickle.dump(roster, f)

with open('computer_science_bktmodel.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

with open('computer_science_roster_model.pkl', 'rb') as f:
    loaded_roster = pickle.load(f)

predictions = loaded_model.predict(data=df)
print(predictions)

# Save predictions to a CSV file
predictions.to_csv('computer_science_csv.csv', index=False)

# from google.colab import files
# files.download('computer_science_predictions.csv')

In [ ]:
# Print the model and roster content
print("Model Content:")
print(model)

print("\nRoster Content:")
# print("Number of Students:", len(roster.students))
print("Skills:", roster.skill_rosters.keys())
print("Mastery State:", roster.mastery_state)
print("Track Progress:", roster.track_progress)
print("Model:", roster.model)

Model Content:
Model(parallel=True, num_fits=1, seed=42, defaults={'skill_name': 'topicSlug', 'correct': 'isCorrect', 'user_id': 'student_id'})

Roster Content:
Skills: dict_keys(['data-structures', 'algorithms', 'operating-systems', 'networking', 'database-systems', 'software-engineering', 'web-development', 'object-oriented-programming', 'machine-learning', 'cloud-computing', 'cybersecurity', 'mobile-development', 'data-analytics', 'software-testing', 'devops', 'design-patterns', 'algorithms-optimization', 'advanced-database-systems', 'distributed-systems', 'artificial-intelligence', 'network-security', 'blockchain', 'user-experience-design', 'ethical-hacking', 'software-architecture', 'quantum-computing', 'big-data'])
Mastery State: 0.95
Track Progress: False
Model: Model(parallel=True, num_fits=1, seed=42, defaults={'skill_name': 'topicSlug', 'correct': 'isCorrect', 'user_id': 'student_id'})


In [ ]:
# We can even define our own metric!
def mae(true_vals, pred_vals):
  """ Calculates the mean absolute error. """
  return np.mean(np.abs(true_vals - pred_vals))

training_mae = model.evaluate(data_path = 'computer_science_csv.csv', metric = mae)
print("Training MAE: %f" % training_mae)

Training MAE: 0.499998


In [ ]:
# training_mae, training_auc, test_mae, test_auc = 0, 0, 0, 0

# skill = 'Calculate unit rate'
# model.fit(data=df, skills = skill)
# training_mae = model.evaluate(data = df, metric = mae)
# training_auc = model.evaluate(data = df, metric = 'auc')
# test_mae = model.evaluate(data = df, metric = mae)
# test_auc = model.evaluate(data = df, metric = 'auc')

In [ ]:
import pandas as pd
import numpy as np
import pyBKT
from pyBKT.models import Model, Roster
from sklearn.model_selection import train_test_split
import pickle
import logging
from typing import Dict, Union, List

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Constants
NUM_STUDENTS = 100
NUM_ATTEMPTS = 100000000
SKILLS = [
    'data-structures', 'algorithms', 'operating-systems', 'networking',
    'database-systems', 'software-engineering', 'web-development',
    'object-oriented-programming', 'machine-learning', 'cloud-computing',
    'cybersecurity', 'mobile-development', 'data-analytics', 'software-testing',
    'devops', 'design-patterns', 'algorithms-optimization',
    'advanced-database-systems', 'distributed-systems', 'artificial-intelligence',
    'network-security', 'blockchain', 'user-experience-design', 'ethical-hacking',
    'software-architecture', 'quantum-computing', 'big-data'
]

# Generate synthetic data
def generate_data(num_students: int, num_attempts: int, skills: list) -> pd.DataFrame:
    """
    Generates synthetic data for training the BKT model.
    """
    logging.info("Generating synthetic data...")
    student_ids = np.random.randint(1, num_students + 1, size=num_attempts)
    topics = np.random.choice(skills, size=num_attempts)
    is_correct = np.random.randint(0, 2, size=num_attempts)

    data = pd.DataFrame({
        'student_id': student_ids,
        'topicSlug': topics,
        'isCorrect': is_correct
    })

    logging.info(f"Generated {len(data)} rows of data.")
    return data

# Train and evaluate the model
def train_and_evaluate_model(data: pd.DataFrame) -> Model:
    """
    Trains and evaluates the BKT model using cross-validation.
    """
    logging.info("Training the BKT model...")
    model = Model(parallel=True, num_fits=1, seed=42, defaults={
        'skill_name': 'topicSlug',
        'correct': 'isCorrect',
        'user_id': 'student_id'
    })

    # Split data into training and validation sets
    train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)

    # Train the model
    model.fit(data=train_data)

    # Evaluate the model on the validation set
    val_predictions = model.predict(data=val_data)
    val_accuracy = np.mean(val_predictions['isCorrect'] == val_predictions['correct_predictions'])
    logging.info(f"Validation Accuracy: {val_accuracy:.2%}")

    return model

# Initialize the roster with skill-specific parameters
def initialize_roster(model: Model, skills: list) -> Roster:
    """
    Initializes the roster with skill-specific parameters.
    """
    logging.info("Initializing the roster...")
    roster = Roster(students=[], skills=skills, model=model)

    # Add skill-specific learning and forgetting rates
    for skill in skills:
        roster.skill_rosters[skill].learns = 0.3  # Default learning rate
        roster.skill_rosters[skill].forgets = 0.05  # Default forgetting rate

    # Set mastery and review thresholds
    roster.mastery_threshold = 0.95
    roster.review_threshold = 0.3

    logging.info("Roster initialized successfully.")
    return roster

# Save predictions to a CSV file
def save_predictions(predictions: pd.DataFrame, filename: str) -> None:
    """
    Saves the model's predictions to a CSV file.
    """
    logging.info(f"Saving predictions to {filename}...")
    predictions.to_csv(filename, index=False)
    logging.info("Predictions saved successfully.")

# Main function
def main():
    # Generate synthetic data
    data = generate_data(NUM_STUDENTS, NUM_ATTEMPTS, SKILLS)

    # Train and evaluate the model
    model = train_and_evaluate_model(data)

    # Initialize the roster
    roster = initialize_roster(model, SKILLS)

    # Save the model and roster using pickle
    logging.info("Saving the model and roster...")
    with open('computer_science_bktmodel.pkl', 'wb') as f:
        pickle.dump(model, f)
    with open('computer_science_roster_model.pkl', 'wb') as f:
        pickle.dump(roster, f)
    logging.info("Model and roster saved successfully.")

    # Load the model and roster using pickle
    logging.info("Loading the model and roster...")
    with open('computer_science_bktmodel.pkl', 'rb') as f:
        loaded_model = pickle.load(f)
    with open('computer_science_roster_model.pkl', 'rb') as f:
        loaded_roster = pickle.load(f)
    logging.info("Model and roster loaded successfully.")

    # Generate predictions
    logging.info("Generating predictions...")
    predictions = loaded_model.predict(data=data)
    save_predictions(predictions, 'computer_science_predictions.csv')

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import pyBKT
from pyBKT.models import Model, Roster
from sklearn.model_selection import train_test_split
import logging
import pickle
from typing import List, Dict, Union

class RealisticLearningDataGenerator:
    def __init__(self, skills: List[str], num_students: int = 500):
        self.skills = skills
        self.num_students = num_students
        np.random.seed(42)
        logging.basicConfig(level=logging.INFO)

    def generate_skill_progression(self) -> Dict:
        """Create realistic skill difficulty and learning parameters."""
        return {
            skill: {
                'initial_difficulty': np.random.uniform(0.6, 0.9),
                'learning_rate': np.random.uniform(0.1, 0.4),
                'forgetting_rate': np.random.uniform(0.01, 0.1)
            } for skill in self.skills
        }

    def generate_student_learning_trajectory(
      self,
      skill_params: Dict,
      attempts_per_skill: Union[int, Dict[str, int]] = None
  ) -> pd.DataFrame:
      """Generate a student's learning trajectory with balanced skill attempts."""
      # Handle different input types for attempts_per_skill
      if attempts_per_skill is None:
          # Default balanced distribution
          total_attempts = 5000000  # Total attempts across all skills
          num_skills = len(self.skills)
          base_attempts = total_attempts // num_skills
          remainder = total_attempts % num_skills

          attempts_per_skill = {
              skill: base_attempts + (1 if i < remainder else 0)
              for i, skill in enumerate(self.skills)
          }
      elif isinstance(attempts_per_skill, int):
          # If a single integer is provided, use same attempts for all skills
          attempts_per_skill = {skill: attempts_per_skill for skill in self.skills}

      attempts = []
      student_id = np.random.randint(1, 10000000)
      current_mastery = {skill: 0 for skill in self.skills}
      streak_counts = {skill: {'correct': 0, 'incorrect': 0} for skill in self.skills}
      streak_multipliers = {skill: 1.0 for skill in self.skills}

      for skill in self.skills:
          skill_attempts = attempts_per_skill[skill]
          skill_difficulty = skill_params[skill]['initial_difficulty']

          for _ in range(skill_attempts):
              # Base change parameters
              base_change = 0.001
              correctness_variance = 0.005
              max_streak_multiplier = 1.03  # 3% hard cap

              # Determine correctness based on current mastery
              correctness_prob = max(0, min(1,
                  current_mastery[skill] * (1 - skill_difficulty) +
                  np.random.normal(0, 0.1)
              ))

              is_correct = 1 if np.random.random() < correctness_prob else 0

              # Update streak logic
              if is_correct:
                  streak_counts[skill]['correct'] += 1
                  streak_counts[skill]['incorrect'] = 0
                  streak_multipliers[skill] = min(
                      max_streak_multiplier,
                      1 + (streak_counts[skill]['correct'] * 0.01)
                  )
              else:
                  streak_counts[skill]['incorrect'] += 1
                  streak_counts[skill]['correct'] = 0
                  streak_multipliers[skill] = max(
                      1.0,
                      1 - (streak_counts[skill]['incorrect'] * 0.01)
                  )

              # Gradual mastery change with streak multiplier
              if is_correct:
                  mastery_change = (base_change + (correctness_variance * 1.5)) * streak_multipliers[skill]
                  current_mastery[skill] += mastery_change * (1 - current_mastery[skill])
              else:
                  mastery_change = (base_change + (correctness_variance * 0.5)) * streak_multipliers[skill]
                  current_mastery[skill] -= mastery_change * current_mastery[skill]

              # Ensure mastery stays within 0-1 range
              current_mastery[skill] = max(0, min(1, current_mastery[skill]))

              attempts.append({
                  'student_id': student_id,
                  'topicSlug': skill,
                  'isCorrect': is_correct,
                  'current_mastery': current_mastery[skill],
                  'mastery_change': mastery_change,
                  'streak_multiplier': streak_multipliers[skill],
                  'correct_streak': streak_counts[skill]['correct'],
                  'incorrect_streak': streak_counts[skill]['incorrect']
              })

      return pd.DataFrame(attempts)

    def generate_dataset(
        self,
        attempts_per_skill: Dict[str, int] = None,
        total_students: int = None
    ) -> pd.DataFrame:
        """Generate comprehensive learning dataset."""
        total_students = total_students or self.num_students
        skill_params = self.generate_skill_progression()

        all_data = []
        for _ in range(total_students):
            student_data = self.generate_student_learning_trajectory(
                skill_params,
                attempts_per_skill
            )
            all_data.append(student_data)

        return pd.concat(all_data, ignore_index=True)

    def train_and_save_models(self, num_attempts_per_student: int = 50):
      """Train BKT model with realistic data and save."""
      # Generate full dataset
      data = self.generate_dataset(num_attempts_per_student)

      # Split data
      train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

      # Train BKT model
      model = Model(parallel=True, defaults={
          'skill_name': 'topicSlug',
          'correct': 'isCorrect',
          'user_id': 'student_id'
      })

      model.fit(data=train_data)

      # Make predictions on test data
      predictions = model.predict(data=test_data)

      # Extract the relevant column from the predictions DataFrame
      test_data['predicted_correct'] = predictions['correct_predictions']

      # Calculate prediction accuracy
      test_data['prediction_correct'] = (test_data['predicted_correct'] >= 0.5).astype(int)
      test_data['accuracy'] = (test_data['prediction_correct'] == test_data['isCorrect']).astype(int)

      # Save datasets and performance metrics
      data.to_csv('full_learning_dataset.csv', index=False)
      train_data.to_csv('train_dataset.csv', index=False)
      test_data.to_csv('test_dataset_with_predictions.csv', index=False)

      # Compute and log skill-wise performance
      skill_performance = test_data.groupby('topicSlug').agg({
          'accuracy': 'mean',
          'isCorrect': 'count'
      }).rename(columns={'accuracy': 'prediction_accuracy', 'isCorrect': 'total_attempts'})
      skill_performance.to_csv('skill_prediction_performance.csv')

      # Create roster
      roster = Roster(students=[], skills=self.skills, model=model)
      for skill in self.skills:
          roster.skill_rosters[skill].learns = 0.3
          roster.skill_rosters[skill].forgets = 0.05

      roster.mastery_threshold = 0.99
      roster.review_threshold = 0.3

      # Save models
      with open('computer_science_bktmodel.pkl', 'wb') as f:
          pickle.dump(model, f)

      with open('computer_science_roster_model.pkl', 'wb') as f:
          pickle.dump(roster, f)

      # Log performance summary
      overall_accuracy = test_data['accuracy'].mean()
      logging.info(f"Realistic models saved. Overall Prediction Accuracy: {overall_accuracy:.2%}")

      return model, roster, test_data

def main():
    SKILLS = [
    'data-structures', 'algorithms', 'operating-systems', 'networking',
    'database-systems', 'software-engineering', 'web-development',
    'object-oriented-programming', 'machine-learning', 'cloud-computing',
    'cybersecurity', 'mobile-development', 'data-analytics', 'software-testing',
    'devops', 'design-patterns', 'algorithms-optimization',
    'advanced-database-systems', 'distributed-systems', 'artificial-intelligence',
    'network-security', 'blockchain', 'user-experience-design', 'ethical-hacking',
    'software-architecture', 'quantum-computing', 'big-data'
    ]

    generator = RealisticLearningDataGenerator(skills=SKILLS)
    generator.train_and_save_models()

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import pyBKT
from pyBKT.models import Model, Roster
from sklearn.model_selection import train_test_split
import logging
import pickle
from typing import List, Dict, Union

class RealisticLearningDataGenerator:
    def __init__(self, skills: List[str], num_students: int = 500):
        self.skills = skills
        self.num_students = num_students
        np.random.seed(42)
        logging.basicConfig(level=logging.INFO)

    def generate_skill_progression(self) -> Dict:
        """Create realistic skill difficulty and learning parameters."""
        return {
            skill: {
                'initial_difficulty': np.random.uniform(0.6, 0.9),
                'learning_rate': np.random.uniform(0.1, 0.4),
                'forgetting_rate': np.random.uniform(0.01, 0.1)
            } for skill in self.skills
        }

    def generate_student_learning_trajectory(
    self,
    skill_params: Dict,
    attempts_per_skill: Union[int, Dict[str, int]] = None
) -> pd.DataFrame:
      """Generate a student's learning trajectory with balanced skill attempts."""
      if attempts_per_skill is None:
          total_attempts = 5000000
          num_skills = len(self.skills)
          base_attempts = total_attempts // num_skills
          remainder = total_attempts % num_skills

          attempts_per_skill = {
              skill: base_attempts + (1 if i < remainder else 0)
              for i, skill in enumerate(self.skills)
          }
      elif isinstance(attempts_per_skill, int):
          attempts_per_skill = {skill: attempts_per_skill for skill in self.skills}

      attempts = []
      student_id = np.random.randint(1, 10000000)
      current_mastery = {skill: 0 for skill in self.skills}
      streak_counts = {skill: {'correct': 0, 'incorrect': 0} for skill in self.skills}
      max_mastery_streak = {skill: 0 for skill in self.skills}

      for skill in self.skills:
          skill_attempts = attempts_per_skill[skill]
          skill_difficulty = skill_params[skill]['initial_difficulty']

          for _ in range(skill_attempts):
              # Base change parameters
              base_change = np.random.uniform(0.001, 0.005)  # 0.1% to 0.5%
              max_mastery_change = 0.01  # 1% hard cap

              correctness_prob = max(0, min(1,
                  current_mastery[skill] * (1 - skill_difficulty) +
                  np.random.normal(0, 0.1)
              ))

              is_correct = 1 if np.random.random() < correctness_prob else 0

              # Update streak logic
              if is_correct:
                  streak_counts[skill]['correct'] += 1
                  streak_counts[skill]['incorrect'] = 0
              else:
                  streak_counts[skill]['incorrect'] += 1
                  streak_counts[skill]['correct'] = 0

              # Gradual mastery change with hard cap
              if is_correct:
                  if current_mastery[skill] >= 1.0:
                      max_mastery_streak[skill] += 1
                      if max_mastery_streak[skill] >= 5:
                          current_mastery[skill] = 0
                          max_mastery_streak[skill] = 0
                  else:
                      mastery_change = min(max_mastery_change, base_change)
                      current_mastery[skill] += mastery_change * (1 - current_mastery[skill])
              else:
                  mastery_change = min(max_mastery_change, base_change)
                  current_mastery[skill] -= mastery_change * current_mastery[skill]

              # Ensure mastery stays within 0-1 range
              current_mastery[skill] = max(0, min(1, current_mastery[skill]))

              attempts.append({
                  'student_id': student_id,
                  'topicSlug': skill,
                  'isCorrect': is_correct,
                  'current_mastery': current_mastery[skill],
                  'mastery_change': mastery_change,
                  'correct_streak': streak_counts[skill]['correct'],
                  'incorrect_streak': streak_counts[skill]['incorrect'],
                  'max_mastery_streak': max_mastery_streak[skill]
              })

      return pd.DataFrame(attempts)

    def generate_dataset(
        self,
        attempts_per_skill: Dict[str, int] = None,
        total_students: int = None
    ) -> pd.DataFrame:
        """Generate comprehensive learning dataset."""
        total_students = total_students or self.num_students
        skill_params = self.generate_skill_progression()

        all_data = []
        for _ in range(total_students):
            student_data = self.generate_student_learning_trajectory(
                skill_params,
                attempts_per_skill
            )
            all_data.append(student_data)

        return pd.concat(all_data, ignore_index=True)

    def train_and_save_models(self, num_attempts_per_student: int = 500):
      """Train BKT model with realistic data and save."""
      data = self.generate_dataset(num_attempts_per_student)

      # Split data
      train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

      # Train BKT model
      model = Model(parallel=True, defaults={
          'skill_name': 'topicSlug',
          'correct': 'isCorrect',
          'user_id': 'student_id'
      })

      model.fit(data=train_data)

      # Make predictions on test data
      predictions = model.predict(data=test_data)

      # Extract the relevant column from the predictions DataFrame
      test_data['predicted_correct'] = predictions['correct_predictions']

      # Calculate prediction accuracy
      test_data['prediction_correct'] = (test_data['predicted_correct'] >= 0.5).astype(int)
      test_data['accuracy'] = (test_data['prediction_correct'] == test_data['isCorrect']).astype(int)

      # Save datasets and performance metrics
      data.to_csv('full_learning_dataset.csv', index=False)
      train_data.to_csv('train_dataset.csv', index=False)
      test_data.to_csv('test_dataset_with_predictions.csv', index=False)

      # Compute and log skill-wise performance
      skill_performance = test_data.groupby('topicSlug').agg({
          'accuracy': 'mean',
          'isCorrect': 'count'
      }).rename(columns={'accuracy': 'prediction_accuracy', 'isCorrect': 'total_attempts'})
      skill_performance.to_csv('skill_prediction_performance.csv')

      roster = Roster(students=[], skills=self.skills, model=model)
      for skill in self.skills:
          roster.skill_rosters[skill].learns = 0.3
          roster.skill_rosters[skill].forgets = 0.05

      roster.mastery_threshold = 0.99
      roster.review_threshold = 0.3

      with open('computer_science_bktmodel-01-26-25.pkl', 'wb') as f:
          pickle.dump(model, f)

      with open('computer_science_roster_model-01-26-25.pkl', 'wb') as f:
          pickle.dump(roster, f)

      # Log performance summary
      overall_accuracy = test_data['accuracy'].mean()
      logging.info(f"Realistic models saved. Overall Prediction Accuracy: {overall_accuracy:.2%}")

      return model, roster, test_data

def main():
    SKILLS = [
    'data-structures', 'algorithms', 'operating-systems', 'networking',
    'database-systems', 'software-engineering', 'web-development',
    'object-oriented-programming', 'machine-learning', 'cloud-computing',
    'cybersecurity', 'mobile-development', 'data-analytics', 'software-testing',
    'devops', 'design-patterns', 'algorithms-optimization',
    'advanced-database-systems', 'distributed-systems', 'artificial-intelligence',
    'network-security', 'blockchain', 'user-experience-design', 'ethical-hacking',
    'software-architecture', 'quantum-computing', 'big-data'
    ]

    generator = RealisticLearningDataGenerator(skills=SKILLS)
    generator.train_and_save_models()

if __name__ == "__main__":
    main()